# Load Base Models - Multi-Model Loader

This notebook loads and validates the base models used across training notebooks.

## Models Available:
| Model | HuggingFace ID | Size (params) | VRAM (4-bit) | VRAM (bf16) |
|-------|----------------|---------------|--------------|-------------|
| Hunyuan-7B | `tencent/Hunyuan-7B-Instruct` | ~7B | ~5GB | ~14GB |
| Qwen2.5-7B | `Qwen/Qwen2.5-7B-Instruct` | ~7B | ~5GB | ~14GB |
| Qwen3-8B | `Qwen/Qwen3-8B` | ~8B | ~6GB | ~16GB |

## Usage:
- Set `QUICK_RUN = True` for fast validation (loads tokenizer only)
- Set `QUICK_RUN = False` to fully load models
- Set `SAVE_MODELS = True` to save loaded models to disk (saved by model name)
- Use `MODELS_TO_LOAD` to select a subset of models
- Configure `OUTPUT_DIR` to specify where models are saved
- Designed for Kaggle H100 (80GB VRAM)

## Model Saving:
Each model is saved to `/kaggle/working/loaded_models/{model_name}/` containing:
- Model weights
- Tokenizer files
- `model_info.json` with model configuration

In [ ]:
!pip install -q -U "transformers>=4.55.0" peft accelerate bitsandbytes datasets scikit-learn joblib

In [ ]:
import os
import gc
import json
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Quick run: only loads tokenizers to validate access (fast)
# Full run: loads complete models with quantization
QUICK_RUN = True

# Use 4-bit quantization (recommended for H100 to fit multiple models)
USE_4BIT = True

# Save loaded models to disk
SAVE_MODELS = True

# Output directory for saved models
OUTPUT_DIR = Path("/kaggle/working/loaded_models")

# HuggingFace token (set via environment or Kaggle secrets)
HF_TOKEN = os.getenv("HF_TOKEN")

# =============================================================================
# MODEL REGISTRY - Add new models here
# =============================================================================

@dataclass
class ModelConfig:
    """Configuration for each model."""
    name: str                    # Display name
    hf_id: str                   # HuggingFace model ID
    params: str                  # Parameter count (display)
    vram_4bit: str               # Estimated VRAM in 4-bit
    vram_bf16: str               # Estimated VRAM in bf16
    local_path: Optional[str]    # Optional local/Kaggle path
    trust_remote_code: bool      # Whether to trust remote code
    target_modules: list         # LoRA target modules

MODEL_REGISTRY = {
    "hunyuan-7b": ModelConfig(
        name="Hunyuan-7B-Instruct",
        hf_id="tencent/Hunyuan-7B-Instruct",
        params="~7B",
        vram_4bit="~5GB",
        vram_bf16="~14GB",
        local_path=None,
        trust_remote_code=True,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj"],
    ),
    "qwen2-7b": ModelConfig(
        name="Qwen2.5-7B-Instruct",
        hf_id="Qwen/Qwen2.5-7B-Instruct",
        params="~7B",
        vram_4bit="~5GB",
        vram_bf16="~14GB",
        local_path="/kaggle/input/qwen-2-5-map/transformers/default/1",
        trust_remote_code=True,
        target_modules=["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ),
    "qwen3-8b": ModelConfig(
        name="Qwen3-8B",
        hf_id="Qwen/Qwen3-8B",
        params="~8B",
        vram_4bit="~6GB",
        vram_bf16="~16GB",
        local_path=None,
        trust_remote_code=True,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    ),
    # -------------------------------------------------------------------------
    # ADD MORE MODELS HERE - Example:
    # -------------------------------------------------------------------------
    # "llama3-8b": ModelConfig(
    #     name="Llama-3-8B-Instruct",
    #     hf_id="meta-llama/Meta-Llama-3-8B-Instruct",
    #     params="~8B",
    #     vram_4bit="~6GB",
    #     vram_bf16="~16GB",
    #     local_path=None,
    #     trust_remote_code=False,
    #     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # ),
}

# =============================================================================
# SELECT WHICH MODELS TO LOAD
# =============================================================================

# Options: "all" or a list like ["hunyuan-7b", "qwen3-8b"]
MODELS_TO_LOAD = "all"  # Load all models
# MODELS_TO_LOAD = ["qwen3-8b"]  # Load only Qwen3-8B
# MODELS_TO_LOAD = ["hunyuan-7b", "qwen2-7b"]  # Load specific subset

# Resolve which models to actually load
if MODELS_TO_LOAD == "all":
    SELECTED_MODELS = list(MODEL_REGISTRY.keys())
else:
    SELECTED_MODELS = MODELS_TO_LOAD

# Create output directory if saving models
if SAVE_MODELS and not QUICK_RUN:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"QUICK_RUN: {QUICK_RUN}")
print(f"USE_4BIT: {USE_4BIT}")
print(f"SAVE_MODELS: {SAVE_MODELS}")
if SAVE_MODELS and not QUICK_RUN:
    print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Models to load: {SELECTED_MODELS}")
print("=" * 60)

In [ ]:
# =============================================================================
# DISPLAY MODEL INFO
# =============================================================================

print("\n" + "=" * 80)
print("AVAILABLE MODELS")
print("=" * 80)
print(f"{'Key':<15} {'Name':<25} {'Params':<10} {'VRAM(4bit)':<12} {'VRAM(bf16)':<12}")
print("-" * 80)

for key, cfg in MODEL_REGISTRY.items():
    selected = "*" if key in SELECTED_MODELS else " "
    print(f"{selected} {key:<13} {cfg.name:<25} {cfg.params:<10} {cfg.vram_4bit:<12} {cfg.vram_bf16:<12}")

print("-" * 80)
print("* = selected for loading")

# Calculate total estimated VRAM
def parse_vram(s: str) -> float:
    """Parse VRAM string like '~5GB' to float."""
    return float(s.replace("~", "").replace("GB", ""))

total_vram_4bit = sum(parse_vram(MODEL_REGISTRY[k].vram_4bit) for k in SELECTED_MODELS)
total_vram_bf16 = sum(parse_vram(MODEL_REGISTRY[k].vram_bf16) for k in SELECTED_MODELS)

print(f"\nEstimated total VRAM for selected models:")
print(f"  4-bit: ~{total_vram_4bit:.0f}GB")
print(f"  bf16:  ~{total_vram_bf16:.0f}GB")

In [ ]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def get_model_path(cfg: ModelConfig) -> str:
    """Get the model path, preferring local path if it exists."""
    if cfg.local_path and Path(cfg.local_path).exists():
        print(f"  Using local path: {cfg.local_path}")
        return cfg.local_path
    print(f"  Using HuggingFace: {cfg.hf_id}")
    return cfg.hf_id


def get_quantization_config():
    """Get BitsAndBytes config for 4-bit quantization."""
    if not USE_4BIT:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )


def load_tokenizer(cfg: ModelConfig) -> AutoTokenizer:
    """Load tokenizer for model."""
    model_path = get_model_path(cfg)
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=cfg.trust_remote_code,
        token=HF_TOKEN,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def load_model(cfg: ModelConfig) -> AutoModelForCausalLM:
    """Load full model with optional quantization."""
    model_path = get_model_path(cfg)
    quantization_config = get_quantization_config()
    
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.bfloat16 if not USE_4BIT else None,
        trust_remote_code=cfg.trust_remote_code,
        token=HF_TOKEN,
    )
    return model


def save_model_and_tokenizer(model, tokenizer, cfg: ModelConfig, model_key: str):
    """Save model and tokenizer to disk using the model's name."""
    if not SAVE_MODELS:
        return None
    
    # Create save directory using model name
    save_path = OUTPUT_DIR / cfg.name
    save_path.mkdir(parents=True, exist_ok=True)
    
    print(f"  Saving to: {save_path}")
    
    # Save tokenizer
    tokenizer.save_pretrained(str(save_path))
    
    # Save model (handle quantized models)
    if USE_4BIT:
        # For quantized models, save the model with its quantization config
        model.save_pretrained(str(save_path), safe_serialization=True)
    else:
        model.save_pretrained(str(save_path))
    
    # Save model config info
    info = {
        "model_key": model_key,
        "model_name": cfg.name,
        "hf_id": cfg.hf_id,
        "params": cfg.params,
        "quantized_4bit": USE_4BIT,
        "target_modules": cfg.target_modules,
    }
    
    with open(save_path / "model_info.json", "w") as f:
        json.dump(info, f, indent=2)
    
    print(f"  Saved successfully!")
    return str(save_path)


def get_model_size(model) -> dict:
    """Calculate model size statistics."""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Estimate memory usage
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    total_size_gb = (param_size + buffer_size) / 1e9
    
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "total_params_str": f"{total_params / 1e9:.2f}B",
        "model_size_gb": f"{total_size_gb:.2f}GB",
    }


def clear_memory():
    """Clear GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [ ]:
# =============================================================================
# LOAD MODELS
# =============================================================================

loaded_models = {}
loaded_tokenizers = {}
model_stats = {}
saved_paths = {}

for model_key in SELECTED_MODELS:
    cfg = MODEL_REGISTRY[model_key]
    print("\n" + "=" * 60)
    print(f"Loading: {cfg.name} ({model_key})")
    print("=" * 60)
    
    try:
        # Always load tokenizer
        print("Loading tokenizer...")
        tokenizer = load_tokenizer(cfg)
        loaded_tokenizers[model_key] = tokenizer
        print(f"  Vocab size: {tokenizer.vocab_size}")
        print(f"  Pad token: {tokenizer.pad_token}")
        
        if QUICK_RUN:
            print("  [QUICK_RUN] Skipping full model load")
            model_stats[model_key] = {
                "status": "tokenizer_only",
                "vocab_size": tokenizer.vocab_size,
            }
        else:
            # Load full model
            print("Loading model...")
            model = load_model(cfg)
            loaded_models[model_key] = model
            
            # Get stats
            stats = get_model_size(model)
            stats["status"] = "loaded"
            stats["vocab_size"] = tokenizer.vocab_size
            model_stats[model_key] = stats
            
            print(f"  Total params: {stats['total_params_str']}")
            print(f"  Model size: {stats['model_size_gb']}")
            
            # Report GPU memory
            if torch.cuda.is_available():
                allocated = torch.cuda.memory_allocated() / 1e9
                reserved = torch.cuda.memory_reserved() / 1e9
                print(f"  GPU memory allocated: {allocated:.2f}GB")
                print(f"  GPU memory reserved: {reserved:.2f}GB")
            
            # Save model and tokenizer
            if SAVE_MODELS:
                print("  Saving model and tokenizer...")
                save_path = save_model_and_tokenizer(model, tokenizer, cfg, model_key)
                saved_paths[model_key] = save_path
                stats["saved_path"] = save_path
        
        print(f"  SUCCESS")
        
    except Exception as e:
        print(f"  FAILED: {e}")
        model_stats[model_key] = {"status": "failed", "error": str(e)}

print("\n" + "=" * 60)
print("LOADING COMPLETE")
print("=" * 60)

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("LOADING SUMMARY")
print("=" * 80)

for model_key, stats in model_stats.items():
    cfg = MODEL_REGISTRY[model_key]
    status = stats.get("status", "unknown")
    
    if status == "loaded":
        print(f"[OK] {cfg.name}: {stats['total_params_str']} params, {stats['model_size_gb']} loaded")
        if "saved_path" in stats:
            print(f"     Saved to: {stats['saved_path']}")
    elif status == "tokenizer_only":
        print(f"[OK] {cfg.name}: Tokenizer loaded (vocab={stats['vocab_size']})")
    else:
        print(f"[FAIL] {cfg.name}: {stats.get('error', 'unknown error')}")

print("\n" + "-" * 80)
print("Available objects:")
print(f"  loaded_tokenizers: {list(loaded_tokenizers.keys())}")
print(f"  loaded_models: {list(loaded_models.keys())}")
print(f"  MODEL_REGISTRY: Configuration for all models")
if saved_paths:
    print(f"  saved_paths: {saved_paths}")

if torch.cuda.is_available():
    print(f"\nFinal GPU memory:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f}GB")
    print(f"  Reserved: {torch.cuda.memory_reserved() / 1e9:.2f}GB")

In [ ]:
# =============================================================================
# OPTIONAL: Test inference with loaded models
# =============================================================================

if not QUICK_RUN and loaded_models:
    print("\nTesting inference...\n")
    
    test_prompt = "What is 2 + 2? Answer:"
    
    for model_key, model in loaded_models.items():
        tokenizer = loaded_tokenizers[model_key]
        cfg = MODEL_REGISTRY[model_key]
        
        print(f"{cfg.name}:")
        inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"  {response}\n")
else:
    print("Skipping inference test (QUICK_RUN=True or no models loaded)")

In [ ]:
# =============================================================================
# CLEANUP (Optional - run if you need to free memory)
# =============================================================================

# Uncomment to clear all loaded models from memory:
# for key in list(loaded_models.keys()):
#     del loaded_models[key]
# clear_memory()
# print("Cleared all models from memory")

## How to Add a New Model

1. Add a new entry to `MODEL_REGISTRY` in the configuration cell:

```python
"your-model-key": ModelConfig(
    name="Display Name",
    hf_id="org/model-name",
    params="~XB",
    vram_4bit="~XGB",
    vram_bf16="~XGB",
    local_path=None,  # or "/kaggle/input/your-model-path"
    trust_remote_code=True,  # or False
    target_modules=["q_proj", "v_proj", ...],  # LoRA targets
),
```

2. Add the model key to `MODELS_TO_LOAD` or use `"all"`

3. Run the notebook to validate the model loads correctly